# Impact of External Ressource on Detection Performance

In [1]:
import sys
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import combinations

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR, EXP_COLS, CLASS_COLS
from utils.analysis_helpers import wrap_model_comparison, wrap_model_comparison_union, print_model_comparison_results, combine_model_comparison_outputs, extract_metric_values, compute_balanced_metrics
from utils.stats_helpers import make_ipw_weights


In [2]:
PROVIDER="gemini"

In [3]:
df_b_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_1.feather"))
df_d_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_1.feather"))
df_b_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_0.feather"))
df_d_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_0.feather"))

In [4]:
df_d_1.columns

Index(['comment_cleaned', 'id', 'classification_tax', 'explanation_tax',
       'classification_tax_ex', 'explanation_tax_ex', 'discourse',
       'comment_level', 'comment_codes_all', 'source_outlet',
       'classification_lexicon', 'explanation_lexicon',
       'classification_ihra_binary', 'classification_ihra_binary_cleaned',
       'classification_ihra_explanation',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation', 'classification_no_kb',
       'classification_no_kb_cleaned', 'explanation_ihra_explanation_sections',
       'explanation_lexicon_chapters', 'explanation_lexicon_chapters_no',
       'explanation_lexicon_sections', 'comment_codes_all_list',
       'comment_codes_all_chapters', 'comment_codes_all_sections',
       'explanation_tax_chapters', 'explanation_tax_ex_chapters',
       'explanation_tax_chapters_no', 'explanation_tax_ex_chapters_no',
       'explanation_tax_sections', 'explanation_tax_ex_sections'],
      dtype='object')

In [5]:
df_b_1.columns

Index(['comment_cleaned', 'id', 'classification_tax', 'explanation_tax',
       'classification_tax_ex', 'explanation_tax_ex', 'keyword',
       'ihra_section_1', 'ihra_section_2', 'ihra_sections',
       'classification_lexicon', 'explanation_lexicon',
       'classification_ihra_binary', 'classification_ihra_binary_cleaned',
       'classification_ihra_explanation',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation', 'classification_no_kb',
       'classification_no_kb_cleaned', 'explanation_ihra_explanation_sections',
       'explanation_lexicon_chapters', 'explanation_lexicon_chapters_no',
       'explanation_lexicon_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'explanation_tax_chapters_no',
       'explanation_tax_ex_chapters_no', 'explanation_tax_sections',
       'explanation_tax_ex_sections'],
      dtype='object')

In [6]:
# DIFFERENT FOR GEMINI
for df in [df_d_1, df_d_0, df_b_1, df_b_0]:
    df.rename({"explanation_ihra_explanation": "explanation_ihra_explanation_cleaned"}, axis=1, inplace=True)

In [7]:
cols_d = ['comment_cleaned', 'discourse', 'comment_level', 
          'comment_codes_all', 'source_outlet'] + EXP_COLS + CLASS_COLS + ['classification_lexicon']

for c in cols_d: 
    if c not in df_d_0.columns:
        print(c)

In [8]:
cols_b = ['comment_cleaned', 'keyword', 'ihra_section_1', 'ihra_section_2'] + EXP_COLS + CLASS_COLS + ['classification_lexicon']

for c in cols_b: 
    if c not in df_b_0.columns:
        print(c)

In [9]:
N_neg_total_d = 39424
N_neg_total_b = 9358

### Merge positive and negative samples

In [10]:
df_d_0["label"] = 0
df_d_1["label"] = 1
df_b_0["label"] = 0
df_b_1["label"] = 1

cols_b.insert(0,"label")
cols_d.insert(0,"label")

In [11]:
df_b = pd.concat([df_b_0[cols_b], df_b_1[cols_b]], ignore_index=True)
df_d = pd.concat([df_d_0[cols_d], df_d_1[cols_d]], ignore_index=True)

In [12]:
df_b.label.value_counts()

label
1    1858
0    1000
Name: count, dtype: int64

In [13]:
df_d.label.value_counts()

label
1    2977
0     992
Name: count, dtype: int64

### Combine datasets

In [14]:
w_neg_d = N_neg_total_d / len(df_d_0)
w_neg_b = N_neg_total_b / len(df_b_0)

df_d["dataset_id"] = "d"
df_b["dataset_id"] = "b"

df_union = pd.concat([df_d, df_b], ignore_index=True)

N_neg_total_by_dataset = {
    "d": N_neg_total_d,
    "b": N_neg_total_b,
}

### Distribution of responses in the positive samples

In [15]:
# Bloomington
tmp = {}
for c in CLASS_COLS + ['classification_lexicon']:
    tmp[c] = df_b_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex,classification_lexicon
Yes,0.643165,0.675996,0.877826,0.832078,0.843380
No,0.321851,0.308396,0.108719,0.163079,0.133477
Unsure,0.034984,0.015608,0.013455,0.004844,0.023143


In [16]:
# Decoding  
tmp = {}
for c in CLASS_COLS + ['classification_lexicon']:
    tmp[c] = df_d_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex,classification_lexicon
No,0.487403,0.498488,0.118912,0.189117,0.126973
Unsure,0.055761,0.026201,0.014780,0.007054,0.021834
Yes,0.456836,0.475311,0.866308,0.803829,0.851192


### Distribution of responses in the negative samples

In [17]:
# Bloomington
tmp = {}
for c in CLASS_COLS + ['classification_lexicon']:
    tmp[c] = df_b_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex,classification_lexicon
No,0.860,0.888,0.80,0.820,0.772
Yes,0.097,0.086,0.18,0.174,0.189
Unsure,0.043,0.026,0.02,0.006,0.039


In [18]:
# Decoding
tmp = {}
for c in CLASS_COLS + ['classification_lexicon']:
    tmp[c] = df_d_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex,classification_lexicon
No,0.723964,0.807460,0.807460,0.850806,0.725806
Unsure,0.243680,0.163306,0.112903,0.057460,0.194556
Yes,0.032356,0.029234,0.079637,0.091734,0.079637


### Compute metrics and CIs on weighted union of both datasets

- B and D have the same weight (0.5)
- CI for recall reflects sampling from population; Ci for precision and F1 reflects sampling from population and 1000er sampling from dataset, thus larger CIs

In [19]:
df_b.label.value_counts()

label
1    1858
0    1000
Name: count, dtype: int64

In [20]:
sample_weight_B, _ = make_ipw_weights(df_b.label, N_neg_total_b)
sample_weight_D, _ = make_ipw_weights(df_d.label, N_neg_total_d)

In [21]:
model_comparison_metrics = compute_balanced_metrics(df_B=df_b, df_D=df_d, N_neg_total_B=N_neg_total_b, N_neg_total_D=N_neg_total_d)

In [22]:
model_comparison_metrics

,model,metric,value,ci_low,ci_high
0,no_kb,precision,0.542539,0.496688,0.598500
1,no_kb,recall,0.550000,0.535940,0.564491
2,no_kb,f1,0.544191,0.518722,0.570581
3,ihra,precision,0.580296,0.531863,0.639182
4,ihra,recall,0.575653,0.561686,0.589799
5,ihra,f1,0.575714,0.549873,0.602782
6,tax,precision,0.471462,0.439799,0.507655
7,tax,recall,0.872067,0.862138,0.881784
8,tax,f1,0.611850,0.583831,0.642086
9,tax_ex,precision,0.442620,0.412961,0.477432


In [23]:
model_comparison_metrics.to_parquet(f"{PROVIDER}_model_comparison_metrics.parquet", index=False)

### Export errors for qualitative analysis

In [24]:
df_b[(df_b["label"]==0) & (df_b["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_decoding.csv"), index=False)
df_b[(df_b["label"]==0) & (df_b["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_decoding.csv"), index=False)

### Statistical model comparison vs baseline (no_kb)

## Bloomington

Mapping "Yes" to 1 and Unsure to 0

1. class proportion as in the original dataset
2. class balance

In [25]:
res_b_weighted = wrap_model_comparison(df=df_b, N_neg_total=N_neg_total_b, with_lex=True)
print_model_comparison_results(res_b_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.6095
metric(no_kb): 0.5683
Diff (model - baseline): 0.0412
95% CI: [0.0035, 0.0791]
p-value (raw): 0.0327
p-value (adjusted): 0.0327
Effect size h: 0.0837 (95% CI: [0.0071, 0.1617])

Model: tax
metric(tax): 0.4919
metric(no_kb): 0.5683
Diff (model - baseline): -0.0764
95% CI: [-0.1153, -0.0403]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.1532 (95% CI: [-0.2324, -0.0806])

Model: tax_ex
metric(tax_ex): 0.4870
metric(no_kb): 0.5683
Diff (model - baseline): -0.0813
95% CI: [-0.1222, -0.0430]
p-value (raw): 0.0001
p-value (adjusted): 0.0002
Effect size h: -0.1630 (95% CI: [-0.2460, -0.0861])

Model: lex
metric(lex): 0.4698
metric(no_kb): 0.5683
Diff (model - baseline): -0.0985
95% CI: [-0.1413, -0.0583]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.1975 (95% CI: [-0.2849, -0.1168])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.6760
metric(no_kb): 0.6432
Diff (model - baseline): 0.0328
95% CI: [

### Decoding

Mapping "Yes" to 1 and "Unsure" to 0

1. class proportion as in the original dataset
2. class balance

In [26]:
res_d_weighted = wrap_model_comparison(df=df_d, N_neg_total=N_neg_total_d, with_lex=True)
print_model_comparison_results(res_d_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.5511
metric(no_kb): 0.5168
Diff (model - baseline): 0.0343
95% CI: [-0.0539, 0.1238]
p-value (raw): 0.441
p-value (adjusted): 0.441
Effect size h: 0.0689 (95% CI: [-0.1089, 0.2509])

Model: tax
metric(tax): 0.4510
metric(no_kb): 0.5168
Diff (model - baseline): -0.0658
95% CI: [-0.1510, 0.0068]
p-value (raw): 0.0706
p-value (adjusted): 0.1983
Effect size h: -0.1317 (95% CI: [-0.3035, 0.0137])

Model: tax_ex
metric(tax_ex): 0.3982
metric(no_kb): 0.5168
Diff (model - baseline): -0.1186
95% CI: [-0.2049, -0.0472]
p-value (raw): 0.0007
p-value (adjusted): 0.0028
Effect size h: -0.2386 (95% CI: [-0.4135, -0.0953])

Model: lex
metric(lex): 0.4466
metric(no_kb): 0.5168
Diff (model - baseline): -0.0701
95% CI: [-0.1606, 0.0067]
p-value (raw): 0.0661
p-value (adjusted): 0.1983
Effect size h: -0.1405 (95% CI: [-0.3228, 0.0135])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.4753
metric(no_kb): 0.4568
Diff (model - baseline): 

### Combined datasets

In [27]:
# Union of both datasets is treated according to the class distribution in the original datasets and according to the overall dataset size, thus decoding matters more
res_union_weighted = wrap_model_comparison_union(df_union, N_neg_total_by_dataset)
print_model_comparison_results(res_union_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.5771
metric(no_kb): 0.5397
Diff (model - baseline): 0.0374
95% CI: [-0.0153, 0.0901]
p-value (raw): 0.1637
p-value (adjusted): 0.1637
Effect size h: 0.0754 (95% CI: [-0.0309, 0.1827])

Model: tax
metric(tax): 0.4660
metric(no_kb): 0.5397
Diff (model - baseline): -0.0736
95% CI: [-0.1253, -0.0259]
p-value (raw): 0.0019
p-value (adjusted): 0.0038
Effect size h: -0.1474 (95% CI: [-0.2517, -0.0517])

Model: tax_ex
metric(tax_ex): 0.4289
metric(no_kb): 0.5397
Diff (model - baseline): -0.1108
95% CI: [-0.1634, -0.0621]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.2221 (95% CI: [-0.3286, -0.1245])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5524
metric(no_kb): 0.5284
Diff (model - baseline): 0.0240
95% CI: [nan, nan]
p-value (raw): 7.215e-06
p-value (adjusted): 7.215e-06
Effect size h: 0.0481 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.8707
metric(no_kb): 0.5284
Diff (model - baseline): 0.3423
95% CI: [

In [28]:
# Union of both datasets is treated according to the class distribution in the original datasets but both datasets weightes equally

res_union_dataset_balanced = combine_model_comparison_outputs(
    {
        "B": res_b_weighted,
        "D": res_d_weighted,
    },
    dataset_weights={
        "B": 0.5,
        "D": 0.5,
    },
)


In [29]:
print_model_comparison_results(res_union_dataset_balanced)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.5803
metric(no_kb): 0.5425
Diff (model - baseline): 0.0378
95% CI: [nan, nan]
p-value (raw): 0.441
p-value (adjusted): 0.441
Effect size h: 0.0763 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.4715
metric(no_kb): 0.5425
Diff (model - baseline): -0.0711
95% CI: [nan, nan]
p-value (raw): 0.0706
p-value (adjusted): 0.1983
Effect size h: -0.1425 (95% CI: [nan, nan])

Model: tax_ex
metric(tax_ex): 0.4426
metric(no_kb): 0.5425
Diff (model - baseline): -0.0999
95% CI: [nan, nan]
p-value (raw): 0.0007
p-value (adjusted): 0.0028
Effect size h: -0.2008 (95% CI: [nan, nan])

Model: lex
metric(lex): 0.4582
metric(no_kb): 0.5425
Diff (model - baseline): -0.0843
95% CI: [nan, nan]
p-value (raw): 0.0661
p-value (adjusted): 0.1983
Effect size h: -0.1690 (95% CI: [nan, nan])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5757
metric(no_kb): 0.5500
Diff (model - baseline): 0.0257
95% CI: [nan, nan]
p-value (raw): 0.01257
p-value (

In [30]:
res_union_dataset_balanced

[('precision',
  {'ihra': {'metric_A': 0.5802959514260504,
    'metric_B': 0.5425389461993237,
    'diff': 0.03775700522672665,
    'effect_size_h': np.float64(0.07628015203627247),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.441),
    'p_value_adj': np.float64(0.441)},
   'tax': {'metric_A': 0.4714621134984248,
    'metric_B': 0.5425389461993237,
    'diff': -0.07107683270089896,
    'effect_size_h': np.float64(-0.14245031782364803),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.0706),
    'p_value_adj': np.float64(0.19830000000000003)},
   'tax_ex': {'metric_A': 0.4426197580955735,
    'metric_B': 0.5425389461993237,
    'diff': -0.09991918810375028,
    'effect_size_h': np.float64(-0.20077401179265153),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,


##  Prediction overlap between models

In [31]:
def intersection_over_union(y_preds, label=1):
    y_preds = [np.asarray(y) for y in y_preds]
    intersection = np.logical_and.reduce([y == label for y in y_preds])
    union = np.logical_or.reduce([y == label for y in y_preds])
    if np.sum(union) == 0:
        return np.nan  # avoid division by zero
    return np.round(np.sum(intersection) / np.sum(union), 2)

### Bloomington

In [32]:
# all 4
intersection_over_union([df_b[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.61)

In [33]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.83
IoU for classification_no_kb_cleaned vs classification_tax: 0.69
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.71
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.72
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.75
IoU for classification_tax vs classification_tax_ex: 0.87


#### Positive samples only

In [34]:
intersection_over_union([
    df_b[df_b.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.66)

In [35]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[df_b.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[df_b.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.86
IoU for classification_no_kb_cleaned vs classification_tax: 0.71
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.75
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.76
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.79
IoU for classification_tax vs classification_tax_ex: 0.9


### Decoding

In [36]:
# all 4
intersection_over_union([df_d[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.41)

In [37]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.7
IoU for classification_no_kb_cleaned vs classification_tax: 0.51
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.54
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.53
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.56
IoU for classification_tax vs classification_tax_ex: 0.85


#### Positive samples only

In [38]:
# all 5
intersection_over_union([df_d[df_d.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.43)

In [39]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[df_d.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[df_d.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.71
IoU for classification_no_kb_cleaned vs classification_tax: 0.52
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.55
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.54
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.58
IoU for classification_tax vs classification_tax_ex: 0.87
